<a href="https://colab.research.google.com/github/code-with-Akshaya/AI-Researcher/blob/main/AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Papers to Production: Deep Research Agent**

### **Problem Statement:** Technical and engineering teams often struggle to move beyond surface-level summaries of research papers, documentation, blogs, and code repositories. Valuable insights remain locked in scattered sources, making it difficult to translate academic knowledge into production-quality decisions.


## **Download the dataset from KaggleHub**

In [ ]:
import kagglehub

# Download latest version of arXiv dataset
path = kagglehub.dataset_download("Cornell-University/arxiv")
print("Path to dataset files:", path)


# **Mount Google drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/273 /content/drive/MyDrive/arxiv_data


# **Data Processing**

## Load the dataset into Pandas

Norimalising the dataset which is in the form of .json into Chunks.Chunk abstracts and full text into semantic units (paragraphs, sections).
Extract metadata (authors, categories) for filtering


In [ ]:
import pandas as pd

# Load only first 100,000 rows for testing
df = pd.read_json(
    "/content/drive/MyDrive/arxiv_data/arxiv-metadata-oai-snapshot.json",
        lines=True,
            chunksize=100000
            )

            # Iterate through chunks
for chunk in df:
    print(chunk.head())
    break  # stop after first chunk




In [ ]:
import pandas as pd

# Create an iterator over chunks
df_iter = pd.read_json(
    "/content/drive/MyDrive/arxiv_data/arxiv-metadata-oai-snapshot.json",
        lines=True,
            chunksize=50000   # adjust chunk size based on memory
            )

            # Get the first chunk as a DataFrame
df = next(df_iter)
print(df.head())


### Exract the abstract safely

In [ ]:
texts = df['abstract'].dropna().tolist()
texts = [t.lower().replace("\n", " ") for t in texts]


## Embedding and Faiss Index

### Installing **FAISS** in colab

In [ ]:
!pip install faiss-cpu


### Import the **Faiss**

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss


### Embedding

In [ ]:
# Embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = embedder.encode(texts, show_progress_bar=True)

# Build FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

## Install Transformers

In [ ]:
!pip install --upgrade transformers accelerate sentencepiece



## Generate Function

### loading the Flan-T5 retrival as the workflow

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Flan-T5 for generation
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Retrieval function using arXiv dataframe
def search_with_metadata(query, k=5):
    results = []
    # Search abstracts for query
    matches = df[df['abstract'].str.contains(query, case=False, na=False)].head(k)
    for _, row in matches.iterrows():
        results.append({
           "title": row['title'],
           "authors": row['authors'],
           "year": row['update_date'],
           "abstract": row['abstract']
                 })
    return results

   # Generation function
def ask_agent_local(query, k=5):
# Retrieve top-k papers
    results = search_with_metadata(query, k)

   # Build context string
    context = "\n\n".join([
                  f"Title: {r['title']}\nAuthors: {r['authors']}\nYear: {r['year']}\nAbstract: {r['abstract']}"
                          for r in results
                              ])

    # Create prompt
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nAnswer clearly and cite sources."

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate output
    outputs = generator.generate(
            inputs["input_ids"],
            max_length=512,
            num_beams=4,
            early_stopping=True
           )

     # Decode tokens back to text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Test the pipeline
print(ask_agent_local("Explain quantum error correction"))




## extend your pipeline with Quick Mode vs Deep Mode


### Retrive Function

In [ ]:
def search_with_metadata(query, mode="quick"):
      # Encode query
     query_embedding = embedder.encode([query])

 # Quick Mode → fewer papers, faster
     if mode == "quick":
        k = 3
 # Deep Mode → more papers, richer context
     else:
        k = 10

 # Search FAISS index
     distances, indices = index.search(query_embedding, k)

 # Collect metadata for top-k results
     results = [metadata[i] for i in indices[0]]
     return results


### Generation Function with Mode

In [ ]:
def ask_agent_local(query, mode="quick"):
    results = search_with_metadata(query, mode=mode)

    context = "\n\n".join([
          f"Title: {r['title']}\nAuthors: {r['authors']}\nYear: {r['update_date']}\nAbstract: {r['abstract']}"
                for r in results
                        ])

    prompt = f"Question: {query}\n\nContext:\n{context}\n\nAnswer clearly and cite sources."

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = generator.generate(
                  inputs["input_ids"],
                  max_length=512 if mode=="quick" else 1024,  # longer answers in Deep Mode
                       num_beams=4,
                       early_stopping=True
                           )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
metadata = df[['title','authors','update_date','abstract']].to_dict(orient="records")
# Quick Mode → fast, concise
print("Quick Mode:\n")
print(ask_agent_local("Explain quantum error correction", mode="quick"))

# Deep Mode → richer, more detailed
print("\nDeep Mode:\n")
print(ask_agent_local("Explain quantum error correction", mode="deep"))


In [ ]:
!pip install streamlit
!streamlit run app.py


In [ ]:
import streamlit as st

st.title("ArXiv Research Assistant")

query = st.text_input("Enter your research question:")
mode = st.radio("Choose mode:", ["quick", "deep"])

if st.button("Ask"):
   answer = ask_agent_local(query, mode=mode)
   st.write("### Answer")
   st.write(answer)

   st.write("### Retrieved Papers")
   results = search_with_metadata(query, mode=mode)
   for r in results:
       st.write(f"**{r['title']}** ({r['year']})")
       st.write(f"Authors: {r['authors']}")
       st.write(r['abstract'])
       st.write("---")
